# ⚡ Train Dora-X2 16-Layer MH-RTU Chat Model on GPU (Google Colab)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Baba01hacker666/doranerual/blob/main/notebooks/train_dora_x2_colab.ipynb)

**Hardware Acceleration**: Uses PyTorch with CUDA & mixed precision (AMP) for ultra-fast GPU training.

**Dora-X2 Specifications:**
- **Architecture**: 16 Layers | 768 Dimension | 12 Recurrent Heads (head_dim=64)
- **Sequence Engine**: Multi-Head Recurrent Trace Units (MH-RTU) with learned channel decay
- **Inference Memory**: Exact $O(1)$ Constant State Memory (<50 KB total across all 16 layers)
- **FeedForward**: Standard SwiGLU MLP (~123.1M Parameters)
- **Tokenization**: Zero-dependency raw UTF-8 byte stream (Vocab size = 256)


## 1. Clone Repository & Install Dependencies


In [ ]:
!git clone https://github.com/Baba01hacker666/doranerual.git
%cd doranerual
!pip install -r requirements.txt


## 2. Check GPU & Verify Dora-X2 Specifications


In [ ]:
import torch
import doraneural as dn
from doraneural.dora_x2 import DoraX2Config

print("PyTorch Version:", torch.__version__)
print("CUDA Available: ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:     ", torch.cuda.get_device_name(0))
    print("Total VRAM (GB):", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
else:
    print("Running on CPU. (Tip: In Colab menu, select Runtime -> Change runtime type -> T4 GPU)")

cfg = DoraX2Config()
print(f"Dora-X2 Specs:   {cfg.n_layers} Layers | {cfg.dim} Dim | {cfg.n_heads} Recurrent Heads")
print(f"Total Model:     {cfg.parameter_count:,} parameters ({cfg.parameter_count/1e6:.1f}M)")


## 3. Train Dora-X2 on GPU with PyTorch AMP
Runs sequence chunk training with AdamW accelerated on CUDA GPU hardware.


In [ ]:
!python research/train_dora_x2.py --device cuda --output-dir checkpoints/dora_x2 --dim 768 --layers 16 --heads 12 --epochs 3 --max-bytes 100000 --lr 0.001 --tag colab_dora_x2_run


## 4. Interactive Conversational Chat Session
Stream responses in real time using the $O(1)$ constant state engine.


In [ ]:
from doraneural.dora_x2 import DoraX2LM, DoraX2ChatSession

checkpoint_path = "checkpoints/dora_x2/colab_dora_x2_run"
print(f"Loading checkpoint: {checkpoint_path}")
model = DoraX2LM.load(checkpoint_path)
session = DoraX2ChatSession(model)

# Interactive streaming chat probe
query = "Hello Dora-X2! How does your 16-layer MH-RTU architecture achieve O(1) state memory?"
print(f"User: {query}\n")
print("Assistant: ", end="", flush=True)
for chunk in session.chat(query, max_tokens=100, stream=True):
    print(chunk, end="", flush=True)
print("\n")


## 5. Save Checkpoints to Google Drive (Optional)


In [ ]:
# Optional: Mount Google Drive and save model weights permanently
# from google.colab import drive
# import shutil
# from pathlib import Path
# drive.mount('/content/drive')
# dest = Path('/content/drive/MyDrive/dora_x2_checkpoints')
# dest.mkdir(parents=True, exist_ok=True)
# shutil.copytree('checkpoints/dora_x2', dest / 'colab_latest', dirs_exist_ok=True)
# print(f'Checkpoint saved to {dest / "colab_latest"}')
